# 📈 Análise de Rentabilidade: Produtos e Canais

## Objetivo de Negócio
Atendendo às diretrizes da Diretoria, esta análise visa responder objetivamente:
1. **Quem são os 'Heróis da Margem'?** (Produtos que garantem a sustentabilidade da empresa).
2. **Quem são os 'Vilões'?** (Produtos que vendem muito, mas corroem o lucro por conta de altos custos ou descontos).

**Metodologia:** Utilizaremos a nossa `Fato de Vendas` (tratada, livre de vazamento temporal) para construir a **Curva ABC** de margem de lucro bruto.

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual (estilo limpo executivo)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carregando a Base Fato
con = duckdb.connect()
df = con.execute("SELECT * FROM '../data/processed/fato_vendas.parquet'").df()
print(f"Dados carregados: {len(df):,} linhas de itens vendidos.")

### 1. Curva ABC de Produtos (Focada em Margem Bruta)
No varejo tradicional, muitos focam na *Receita*. Vamos focar no *Lucro Bruto* (`gross_margin`).

In [ ]:
# Agrupando por Produto (SKU)
df_prod = df.groupby(['sku', 'product_name']).agg({
    'item_revenue': 'sum',
    'total_cost': 'sum',
    'gross_margin': 'sum',
    'quantity': 'sum'
}).reset_index()

# Calculando a Margem Bruta Percentual (%)
df_prod['margin_pct'] = (df_prod['gross_margin'] / df_prod['item_revenue']) * 100

# Ordenando pelos que mais geram DINHEIRO (Massa de Margem)
df_prod = df_prod.sort_values('gross_margin', ascending=False)

# Calculando o percentual acumulado (Regra 80/20)
df_prod['margin_cumsum_pct'] = df_prod['gross_margin'].cumsum() / df_prod['gross_margin'].sum() * 100

# Classificando A, B e C
def classify_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

df_prod['curva'] = df_prod['margin_cumsum_pct'].apply(classify_abc)

# Visualizando o Top 10 Produtos - Nossos "Heróis"
display(df_prod.head(10).style.format({
    'item_revenue': 'R$ {:,.2f}',
    'gross_margin': 'R$ {:,.2f}',
    'margin_pct': '{:.1f}%'
}))

### 2. Identificando os "Vilões" da Rentabilidade
Queremos produtos da Curva A em Receita (vendem muito em volume), mas que deixam pouquíssima Margem (< 5%) ou até Margem Negativa.

In [ ]:
# Filtrando produtos com alta receita, mas margem baixa
high_revenue_threshold = df_prod['item_revenue'].quantile(0.80) # Top 20% em vendas brutas
viloes = df_prod[(df_prod['item_revenue'] >= high_revenue_threshold) & (df_prod['margin_pct'] < 10)].sort_values('margin_pct')

if len(viloes) > 0:
    print(f"Foram encontrados {len(viloes)} produtos com alto faturamento e margem preocupante (<10%).")
    display(viloes.head(10))
else:
    print("Não temos 'vilões' críticos na margem base (sem considerar devoluções)!")

### 3. Exemplo Prático do Framework de Decisão
*(Preencha de acordo com os resultados das células acima)*

**Fato Observado:** 
*(Descrever o achado principal da curva ABC ou dos vilões)*

**Hipótese:**
*(Descrever por que esse produto performa assim)*

**Recomendação:**
*(Como a Marina/Sr. Almir devem atuar amanhã para corrigir isso)*